In [0]:
from pyspark.sql.functions import col, sum,count,expr,trim,lower, when


In [0]:
#read data from delta table
df_silver = spark.table("training_catalog.bronze.ecommerce_bronze")

In [0]:
#schema validation
df_silver.printSchema()

In [0]:
#count the ligne
print("Number of rows:", df_silver.count())
print("Number of columns:", len(df_silver.columns))

# Data quality checks

In [0]:
##check null colomm
df_silver.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df_silver.columns
    ]
).show()

In [0]:
# DATA QUALITY CHECK 2 - EXACT DUPLICATES
total_rows = df_silver.count()
unique_rows = df_silver.dropDuplicates().count()

duplicate_rows = total_rows - unique_rows

print("Total rows     :", total_rows)
print("Unique rows    :", unique_rows)
print("Duplicate rows :", duplicate_rows)

In [0]:
from pyspark.sql.functions import count, col
# CHECK DUPLICATE ORDER IDs

df_duplicate_orders = (
    df_silver
    .groupBy("order_id")
    .agg(count("*").alias("row_count"))
    .filter(col("row_count") > 1)
)

display(df_duplicate_orders)

#date verifications

In [0]:
display(
    df_silver.select(
        "order_date",
        "ship_date"
    ).limit(20)
)

In [0]:
df_date_check = df_silver.withColumn(
    "order_date_check",
    expr("""
        coalesce(
            try_to_date(order_date, 'dd-MM-yyyy'),
            try_to_date(order_date, 'M/d/yyyy')
        )
    """)
)

In [0]:
#Maintenant affichons seulement les valeurs problématiques :
display(
    df_date_check
    .filter(
        col("order_date").isNotNull() &
        col("order_date_check").isNull()
    )
    .select(
        "order_id",
        "order_date",
        "ship_date"
    )
)

In [0]:
## DATA QUALITY - COUNT INVALID ORDER DATES
invalid_date_count = (
    df_date_check
    .filter(
        col("order_date").isNotNull() &
        col("order_date_check").isNull()
    )
    .count()
)

total_count = df_silver.count()

print("Total rows        :", total_count)
print("Invalid date rows :", invalid_date_count)

In [0]:

# DATA QUALITY GATE


total_rows = df_silver.count()

invalid_rows = (
    df_date_check
    .filter(
        col("order_date").isNotNull() &
        col("order_date_check").isNull()
    )
    .count()
)

invalid_percentage = (invalid_rows / total_rows) * 100

print("Total rows       :", total_rows)
print("Invalid rows     :", invalid_rows)
print("Invalid rate (%) :", round(invalid_percentage, 2))

In [0]:

#######DATA QUALITY GATE
#######c est quoi le controle ndata quality, je n est pas compris Stop the pipeline if too many invalid records are detected
# DATA QUALITY GATE
# Stopper le pipeline si le pourcentage de dates invalides est trop élevé

QUALITY_THRESHOLD = 4.0

# Calcul du vrai pourcentage de dates invalides
invalid_percentage = (invalid_date_count / total_count) * 100

print(f"Invalid dates: {invalid_percentage:.2f}%")

if invalid_percentage > QUALITY_THRESHOLD:
    raise Exception(
        f"DATA QUALITY FAILED: "
        f"{invalid_percentage:.2f}% invalid dates "
        f"(threshold: {QUALITY_THRESHOLD}%)"
    )
else:
    print(
        f"DATA QUALITY PASSED: "
        f"{invalid_percentage:.2f}% invalid dates"
    )
   

In [0]:
from pyspark.sql import Row

quality_report = [
    Row(
        check_name="Total records",
        failed_records=0,
        total_records=total_count
    ),
    Row(
        check_name="Invalid order dates",
        failed_records=invalid_date_count,
        total_records=total_count
    )
]

df_quality_report = spark.createDataFrame(quality_report)

display(df_quality_report)

In [0]:

df_quarantine = (
    df_date_check
    .filter(
        col("order_date").isNotNull() &
        col("order_date_check").isNull()
    )
)

In [0]:
df_valid = (
    df_date_check
    .filter(
        col("order_date_check").isNotNull()
    )
)

In [0]:
print("Bronze records     :", df_silver.count())
print("Valid records      :", df_valid.count())
print("Quarantine records :", df_quarantine.count())

In [0]:
(
    df_quarantine.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "training_catalog.bronze.ecommerce_quarantine"
    )
)

In [0]:
# REMOVE EXACT DUPLICATES
df_clean = df_valid.dropDuplicates()

In [0]:
before = df_valid.count()
after = df_clean.count()

print("Before deduplication :", before)
print("After deduplication  :", after)
print("Duplicates removed   :", before - after)

In [0]:
#Conversion des types numériques
df_typed = (
    df_clean
    .withColumn(
        "sales_per_order_clean",
        expr("try_cast(sales_per_order AS DOUBLE)")
    )
    .withColumn(
        "order_quantity_clean",
        expr("try_cast(order_quantity AS INT)")
    )
)

In [0]:
#Vérifions les conversions
display(
    df_typed.select(
        "sales_per_order",
        "sales_per_order_clean",
        "order_quantity",
        "order_quantity_clean"
    ).limit(20)
)

######CLEAN SHIP DATE
######Purpose:
######Convert ship_date from STRING to DATE using
#####the formats observed in the source data.
#####Invalid values will become NULL.

In [0]:


df_dates_clean = (
    df_typed
    .withColumn(
        "ship_date_clean",
        expr("""
            coalesce(
                try_to_date(ship_date, 'dd-MM-yyyy'),
                try_to_date(ship_date, 'M/d/yyyy')
            )
        """)
    )
)

In [0]:
display(
    df_dates_clean.select(
        "order_date",
        "order_date_check",
        "ship_date",
        "ship_date_clean"
    ).limit(20)
)

In [0]:
df_dates_clean.select(
    "order_date_check",
    "ship_date_clean"
).printSchema()

#######CLEAN STRING COLUMNS
#######Purpose:
#######Remove unnecessary spaces and standardize text values.

In [0]:
df_strings_clean = (
    df_dates_clean
    .withColumn(
        "customer_country_clean",
        trim(col("customer_country"))
    )
    .withColumn(
        "delivery_status_clean",
        lower(trim(col("delivery_status")))
    )
    .withColumn(
        "customer_segment_clean",
        trim(col("customer_segment"))
    )
)

In [0]:
#Puis vérification des conversions
display(
    df_strings_clean.select(
        "customer_country",
        "customer_country_clean",
        "delivery_status",
        "delivery_status_clean",
        "customer_segment",
        "customer_segment_clean"
    ).limit(20 )
)

######CREATE CALCULATED COLUMN
#####Purpose:
#####Create a business indicator based on profit.

In [0]:
df_enriched = (
    df_strings_clean
    .withColumn(
        "profit_status",
        when(
            col("profit_per_order") > 0,
            "Profitable"
        ).otherwise("Not Profitable")
    )
)

#####BUILD FINAL SILVER DATAFRAME
#####Purpose:
######Keep cleaned columns only and give them final business names.

In [0]:
df_silver_final = df_enriched.select(
    "customer_id",
    "customer_first_name",
    "customer_last_name",

    col("customer_country_clean").alias("customer_country"),
    "customer_city",
    "customer_state",
    "customer_region",
    col("customer_segment_clean").alias("customer_segment"),

    "category_name",
    "product_name",

    "order_id",
    col("order_date_check").alias("order_date"),
    col("ship_date_clean").alias("ship_date"),

    "shipping_type",
    col("delivery_status_clean").alias("delivery_status"),

    col("order_quantity_clean").alias("order_quantity"),
    col("sales_per_order_clean").alias("sales_per_order"),

    "profit_per_order",
    "profit_status",

    "ingest_file_name",
    "ingest_time"
)

In [0]:
df_silver_final.printSchema()

In [0]:
display(df_silver_final.limit(20))

In [0]:
# WRITE CLEAN DATA TO SILVER DELTA TABLE
(
    df_silver_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "training_catalog.silver.ecommerce_clean"
    )
)

In [0]:
%sql
/*verifications sql*/
SELECT *
FROM training_catalog.silver.ecommerce_clean
LIMIT 20;

In [0]:
%sql
DESCRIBE TABLE training_catalog.silver.ecommerce_clean;